### Instructions
- This notebook reproduces the images shown in the preprint [Saint Fleur et al (2025)](https://doi.org/10.5194/egusphere-2025-4244). It is a clik and go, but we encourage you to check if it runs on your system first. You may need to have downloaded the related data_paper first, then jump to the *TODO* flags in the cells below and get them adapted properly.

- To run all in one shot, once ready, click on the **Run All** or **>>** button above

[Note] Please, note that this notebook is not totally optimized, there are many redundancies and scripts that can be simplified. It is only intended to keep the track of the images shown in the related paper

In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from post_process_v2.pp_utils import make_spread_skill_ratio, read_basin_list_from_file
from post_process_v2.run_posp import run_pp

In [ ]:
# %pip install HydroErr pyarrow

## General Options

In [ ]:
DEBUG = True                            #TODO 1: Turn True of False to check if all is ok
CASE_CAMELS = "fr"                      #TODO 2: Choose between "fr" and "us" for independent processing
PATH_TO_DATA_PAPER= r"Y:\repo_egu24"    #TODO 3: adapt to your reality

In [ ]:
GENERAL_ = {"period": {"fr":("20180901","20210831"),
                       "us": ("19891001","19910930")
                       },
             "ensemble_cases": {"fr": ["clim", "hind", "real"],
                              "us": ["clim", "hind"]
                                },
             "context_list":{"fr": ["lstm", "no_in_mlp"], 
                            "us": ["lstm", "sacsma", "no_in_mlp", "lstm_in_mlp", "sma_in_mlp", "predict_lstm_error", "predict_sma_error"]
                            },
            "data_root":fr"{PATH_TO_DATA_PAPER}\data_paper\data",
            "processed_root": fr"{PATH_TO_DATA_PAPER}\data_paper\processed",
            "metrics_root": fr"{PATH_TO_DATA_PAPER}\data_paper\metrics",
            }

In [ ]:

DATA_DIR = GENERAL_.get("data_root") + f"/camels{CASE_CAMELS}"
PROCESSED_DIR = GENERAL_.get("processed_root") + f"/{CASE_CAMELS}"
METRICS_DIR = GENERAL_.get("metrics_root") + f"_temp/{CASE_CAMELS}"  #TODO: You may need to adapt path to save metrics
if DEBUG:
    METRICS_DIR = METRICS_DIR.replace("metrics", "metrics_DEBUG")
    os.makedirs(METRICS_DIR, exist_ok=True)

PERIOD = GENERAL_.get("period").get(CASE_CAMELS)
CONTEXT_LIST = GENERAL_.get("context_list").get(CASE_CAMELS)
ENSEMBLE_CASES = GENERAL_.get("ensemble_cases").get(CASE_CAMELS)

BASINS_FULL = read_basin_list_from_file(DATA_DIR + "/basins_list")
BASINS_56 = read_basin_list_from_file(DATA_DIR +"/basins_56")
HP_LIST = [1, 2, 3, 4, 5, 6, 7]

In [ ]:
def read_pqt(f:str):
    """Read a .parquet file"""
    return pd.read_parquet(f, engine="pyarrow")

## Observed and naive data

In [ ]:
obs_naive = pd.read_parquet(PROCESSED_DIR+"/obs_naive.parquet.gzip", engine="pyarrow").swaplevel(0, 1).sort_index()
obs_naive.head()

## Perfect forecast scenarios processing

In [ ]:
bob_perf = glob(PROCESSED_DIR + f"/*_hp*_PERF{len(BASINS_FULL)}.*")
re_level = ['context', "basin", 'hp', 'year', "seed", 'Date']

for cx in CONTEXT_LIST:
    cx_path = METRICS_DIR + f"/postp_perf/{cx}"
    os.makedirs(cx_path, exist_ok=True)
    all_perf_pred = []
    cx_case = [f for f in bob_perf if cx + "_hp" in f]
    perf_bar = tqdm(cx_case, desc="Reading By-Perf-File", leave=True)
    for i_perf in perf_bar:
        perf_bar.set_postfix_str(f"File: {i_perf}")
        ex_perf = pd.read_parquet(i_perf, engine="pyarrow")["pred"].to_frame("prediction")
        if "seed" in ex_perf.columns:
            ex_perf = ex_perf.set_index("seed", append=True)
        ex_perf = ex_perf.reorder_levels(re_level).sort_index().astype(np.float32)
        all_perf_pred.append(ex_perf)
    all_perf_pred = pd.concat(all_perf_pred, axis=0)
    print("End concat")
    all_perf_pred = all_perf_pred.join(obs_naive, on=["hp", "basin", "Date"], how="inner").sort_index()
    print("compute metric")
    run_pp(path_to=cx_path, pred_all=all_perf_pred, debug=DEBUG)
    print(f"Done on {cx}!")
    del all_perf_pred, cx_path

## Ensemble-based forecasts scenario processing

### Natural Climatology of the discharge

In [ ]:
# Using period and forcing data, compute climatology of the discharge
if len(glob(PROCESSED_DIR+"/natural_*_hp*.parquet.gzip")) == 0:
    from post_process_v2.pp_utils import make_rank_for_basin
    bv_q_clim = []
    for basin in tqdm(BASINS_56):
        df_x = pd.read_csv(DATA_DIR +  f"/all_sim_obs_lstm/{basin}.txt", sep=";", index_col="Date", parse_dates=True)
        rk, mb_q = make_rank_for_basin(df_0=df_x, period=PERIOD, feats=["Q_Obs"])
        mb_q.columns = ["Obs"] + [(i+1) for i in range(mb_q.shape[1]-1)]
        mb_q.index.name="Date"
        mb_q = mb_q.melt(id_vars="Obs", var_name="year", value_name="pred", ignore_index=False).set_index("year", append=True).sort_index()[["pred"]]
        extra_index = pd.MultiIndex.from_tuples([("natural_records", f"{basin}".zfill(8), 0)]*mb_q.shape[0], names=["context", "basin", "seed"])
        mb_q = mb_q.set_index(extra_index, append=True)
        bv_q_clim.append(mb_q)
    bv_q_clim = pd.concat(bv_q_clim, axis=0).reorder_levels(["context", "basin", "Date", "seed", "year"]).sort_index()
    for hp in HP_LIST:
        ne_bv_q = bv_q_clim.set_index(pd.MultiIndex.from_tuples([(hp,)]*bv_q_clim.shape[0], names=["hp"]), append=True)
        ne_bv_q = ne_bv_q.reorder_levels(["context", "hp", "basin", "Date", "seed", "year"]).sort_index()
        ne_bv_q.to_parquet(PROCESSED_DIR+f"/natural_records_hp{hp}_CLIM56.parquet.gzip", compression="gzip", engine="pyarrow")

## Spread skill of the ensemble 

In [ ]:
for case_p in ENSEMBLE_CASES:      
    Cx_LIST = CONTEXT_LIST if "clim" not in case_p else CONTEXT_LIST+["natural_records"]
    for ctx in Cx_LIST:
        cx_path = METRICS_DIR+f"/postp_{case_p}/{ctx}"
        os.makedirs(cx_path, exist_ok=True)
        case_spr = []
        case_e = glob(PROCESSED_DIR+ f"/{ctx}_hp*_{case_p.upper()}56.parquet.gzip")
        case_bar =tqdm(case_e, desc=f"Reading By-{case_p} -- {ctx}")
        for f in case_bar:
            case_bar.set_postfix_str(f"File : {f}")
            pf = pd.read_parquet(f, engine="pyarrow").astype(np.float32)["pred"].to_frame("prediction")
            if "perf" not in case_p:
                pf = pf.loc[pf.index.get_level_values("year")!=-1]
            spr, skl, spread_skill_rate = make_spread_skill_ratio(pf, obs_naive[["obs"]])
            case_spr.append(pd.concat([spr, skl, spread_skill_rate], axis=1))
        case_spr = pd.concat(case_spr, axis=0).sort_index().astype(np.float32)
        case_spr.columns = ["spread", "skill", "ssr"]
        case_spr.to_parquet(cx_path +"/pp_spread_skill_pandas.parquet.gzip", engine="pyarrow", compression="GZIP")
        print(f"Done on {case_p} {ctx}")

## FULL CLIM Process

In [ ]:
# Process each ensemble-based cases sequentially [clim, hind, real]
for case_ens in ENSEMBLE_CASES:
    bob_clim = glob(PROCESSED_DIR + f"/*{case_ens}56.*")
    re_level = ['context', "basin", 'hp', 'year', "seed", 'Date']
    Cx_LIST = CONTEXT_LIST if "clim" not in case_ens else CONTEXT_LIST+["natural_records"]
    for cx in Cx_LIST:
        cx_path = METRICS_DIR+f"/postp_{case_ens}/{cx}"
        os.makedirs(cx_path, exist_ok=True)
        all_ens_proc = []
        cx_case = [f for f in bob_clim if cx+"_hp" in f]
        bar_cases = tqdm(cx_case, desc="by-Cx-case", leave=True)
        for i_clim in bar_cases:
            bar_cases.set_postfix_str(f"File : {i_clim}")
            ex_clim = pd.read_parquet(i_clim, engine="pyarrow")["pred"].to_frame("prediction")
            if "seed" in ex_clim.columns:
                ex_clim = ex_clim.set_index("seed", append=True)
            ex_clim = ex_clim.reorder_levels(re_level).sort_index().astype(np.float32)
            ex_clim = ex_clim.loc[ex_clim.index.get_level_values("year") != -1]
            all_ens_proc.append(ex_clim)
        all_ens_proc = pd.concat(all_ens_proc, axis=0)
        print("End concat")
        all_ens_proc = all_ens_proc.join(obs_naive, on=["hp", "basin", "Date"], how="inner").sort_index()
        print("compute metric")
        run_pp(path_to =cx_path , pred_all=all_ens_proc, debug=DEBUG)
        print(f"Done on {cx}!")
        del all_ens_proc,cx_path

## Grouping metrics

In [ ]:
def _make_grouped_dir(dir_multi):
    """Make a unified dir by adding _grouped to the path of the multi_fold_path"""
    p = Path(dir_multi)
    old = str(p.parent.parent)
    new = str(p.parent.parent.with_name(p.parent.parent.name + "_grouped"))
    dir_uni = str(p).replace(old, new)
    return dir_uni

In [ ]:
av_metrics = ["briers_ie", "briers", "crps", "metrics", "ranks", "rocs", "roc_ie", "spread_skill"]
# Group the scores by context runs
multi_crit = ["perf"] + ENSEMBLE_CASES
for post_x in multi_crit:
    path_pp_multi = METRICS_DIR+f"/postp_{post_x}"
    path_pp_uni = _make_grouped_dir(path_pp_multi)
    os.makedirs(path_pp_uni, exist_ok=True)
    for mx in av_metrics:
        try:
            pp_multi_cx = glob(path_pp_multi +f"/*/pp_{mx}_pandas*gzip")
            d_pmx = pd.concat([pd.read_parquet(pmx, engine="pyarrow") for pmx in pp_multi_cx], axis=0).sort_index()
            d_pmx.to_parquet(path_pp_uni+f"/pp_{mx}_pandas.parquet.gzip", compression="gzip", engine="pyarrow")
        except ValueError as e:
            print(e)
            continue

---
---